# Step 3: Look at the data (EDA)
*Same as `src/03_eda.py`; charts are shown here instead of saved.*

Go to the project folder.

In [ ]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")          # work from the project folder

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

df = pd.read_parquet("data/features/noxi_engagement.parquet")
feature_cols = []
for c in df.columns:
    if c.startswith("voice_") or c.startswith("face_"):
        feature_cols.append(c)
print("windows:", len(df))

**How much data per language**, and **average engagement per language**.

In [ ]:
print(df["language"].value_counts())
print(df.groupby("language")["engagement"].mean())

**ANOVA:** is the average engagement different between the languages? (p < 0.05 = yes)

In [ ]:
english = df[df["language"] == "English"]["engagement"]
french = df[df["language"] == "French"]["engagement"]
german = df[df["language"] == "German"]["engagement"]
f_value, p_value = stats.f_oneway(english, french, german)
print("F =", round(f_value, 2), " p =", p_value)

**Charts:** distribution, by language, by role.

In [ ]:
sns.histplot(df["engagement"], bins=30, kde=True)
plt.title("Engagement distribution")
plt.show()

sns.boxplot(data=df, x="language", y="engagement")
plt.title("Engagement by language")
plt.show()

sns.boxplot(data=df, x="role", y="engagement")
plt.title("Engagement by role")
plt.show()
print(df.groupby("role")["engagement"].mean())

**Which features go with engagement?** Correlation of every feature with engagement.

In [ ]:
corr = df[feature_cols].corrwith(df["engagement"]).abs()
corr = corr.sort_values(ascending=False)
corr.head(10).plot(kind="barh")
plt.title("Top 10 features")
plt.show()

**Voice vs face:** average correlation of each group.

In [ ]:
voice_cols = []
face_cols = []
for c in feature_cols:
    if c.startswith("voice_"):
        voice_cols.append(c)
    else:
        face_cols.append(c)
print("voice:", round(corr[voice_cols].mean(), 3))
print("face: ", round(corr[face_cols].mean(), 3))

**PCA:** squeeze all features into 2 numbers and colour each window by language.

In [ ]:
X = StandardScaler().fit_transform(df[feature_cols].fillna(0))
points = PCA(n_components=2).fit_transform(X)
sns.scatterplot(x=points[:, 0], y=points[:, 1], hue=df["language"], s=8, alpha=0.5)
plt.title("PCA by language")
plt.show()